In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
            
    main_features.append(f'ME_prod_M*_E*')
    main_features.append(f'ME_ratio_M*_E*')
            
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    
    main_features.append(f'MI_prod_M*_I*')
    main_features.append(f'MI_ratio_M*_I*')
    main_features.append(f'MI_spread_M*_I*')
        
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    
    main_features.append(f'MP_prod_M*_P*')
    main_features.append(f'MP_ratio_M*_P*')
    
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
                
    main_features.append(f'MV_ratio_M*_V*')
    main_features.append(f'MV_prod_M*_V*')
        
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.append(f'MS_prod_M*_S*')
    main_features.append(f'MS_weighted_M*_S*')
        
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.append(f'EI_diff_E*_I*')
    main_features.append(f'EI_ratio_E*_I*')
    main_features.append(f'EI_prod_E*_I*')
        
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.append(f'EP_prod_E*_P*')
    main_features.append(f'EP_ratio_E*_P*')
        
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.append(f'EV_prod_E*_V*')
    main_features.append(f'EV_uncertainty_E*_V*')
        
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.append(f'ES_prod_E*_S*')
    main_features.append(f'ES_diff_E*_S*')
        
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.append(f'IP_prod_I*_P*')
    main_features.append(f'IP_discount_I*_P*')
    
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
        
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
        
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.append(f'PV_prod_P*_V*')
    main_features.append(f'PV_stability_P*_V*')
        
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.append(f'PS_prod_P*_S*')
    main_features.append(f'PS_contrarian_P*_S*')
        
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.append(f'VS_prod_V*_S*')
    main_features.append(f'VS_panic_V*_S*')

    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 

    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 

    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 

    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']

    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()

    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    return data

train_processed = preprocessing(train, "train")

train_x = train_processed.drop(columns=["target"])
train_y = train_processed['target']

In [3]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,IP_discount_I*_P*,IV_prod_I*_V*,IS_prod_I*_S*,PV_prod_P*_V*,PV_stability_P*_V*,PS_prod_P*_S*,PS_contrarian_P*_S*,VS_prod_V*_S*,VS_panic_V*_S*,target
6969,0.682135,0.017196,0.007937,0.007937,0.007937,0.007937,0.048280,1.148054,1.303514,1.046752,...,0.415121,39.293238,13.819007,18.370000,0.747024,6.460530,-6.460530,8.648361,8.648361,0.001145
6970,0.681101,0.016865,0.007606,0.007606,0.007606,0.007606,0.008267,1.146588,1.303443,1.047816,...,0.720893,36.652103,17.124485,29.821070,1.341544,13.932911,-13.932911,10.385726,10.385726,0.004738
6971,0.680068,0.016534,0.007275,0.007275,0.007275,0.007275,0.007937,1.145124,1.303371,1.048881,...,0.813072,38.321956,22.500644,35.111530,1.485426,20.615650,-20.615650,13.878609,13.878609,0.006016
6972,0.679035,0.016204,0.006944,0.006944,0.006944,0.006944,0.007606,1.120467,2.311946,1.049948,...,0.707903,33.359403,25.236451,26.727384,1.382853,20.219316,-20.219316,14.621450,14.621450,0.001414
6973,0.678003,0.015873,0.006614,0.006614,0.006614,0.006614,0.007275,1.119052,2.308384,1.051017,...,0.671389,32.548385,21.987350,24.789544,1.295489,16.746035,-16.746035,12.926420,12.926420,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,1.565379,0.184524,0.019180,0.019180,0.005952,0.005952,0.911376,-0.083496,-0.572447,0.223638,...,0.975153,14.998988,12.413334,18.013558,1.492962,14.908227,-14.908227,9.985670,9.985670,0.002457
8986,1.562946,0.184193,0.018849,0.018849,0.005622,0.005622,0.911706,-0.083542,-0.572080,0.222910,...,1.068629,13.720627,21.347952,18.137762,1.714749,28.220582,-28.220582,16.457558,16.457558,0.002312
8987,1.560520,0.183862,0.018519,0.018519,0.005291,0.005291,0.912037,-0.083874,-0.572016,0.222211,...,0.779983,11.527504,17.911727,11.147219,1.459002,17.320831,-17.320831,11.871701,11.871701,0.002891
8988,1.558102,0.183532,0.018188,0.018188,0.004960,0.004960,0.912368,-0.084206,-0.571952,0.221513,...,1.205378,12.385137,7.635535,18.450361,2.161599,11.374792,-11.374792,5.262211,5.262211,0.008310


In [4]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
        
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_mape = mean_absolute_error(train_y, current_train_pred)
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}


LGBM_R_parm = {
               'boosting_type': 'gbdt', 
               'colsample_bytree': 0.95, 
               'learning_rate': 0.08,
               'max_bin': 100,
               'max_depth': 12,
               'metric': 'regression_l2', 
               'min_child_samples': 60,
               'min_data_in_leaf': 20, 
               'n_estimators': 7000,
               'num_leaves': 50,
               'objective': 'mean_absolute_error', 
               'reg_alpha': 0.8,
               'reg_lambda': 3.5, 
               'subsample': 0.75, 
               'verbosity': -1
              }

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y)

ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAPE: 0.0018
Training Model 2...
Cumulative Training MAPE: 0.0017
Training Model 3...
Cumulative Training MAPE: 0.0016
Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
MAPE: 92397018.6396
MAE: 0.0016
RMSE: 0.0042


In [5]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, Bounds
from collections import deque

SIGNAL_MULTIPLIER = 50000.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

class AdaptiveSignalProcessor:
    """
    Advanced signal processor that adapts to market conditions and volatility
    """
    def __init__(self, lookback=180, target_vol_ratio=1.2):
        self.lookback = lookback
        self.target_vol_ratio = target_vol_ratio
        self.prediction_history = deque(maxlen=lookback)
        self.return_history = deque(maxlen=lookback)
        self.vol_ewma = None
        self.alpha = 0.94  # EWMA decay factor
        
    def update_history(self, prediction, actual_return=None):
        """Update historical data"""
        self.prediction_history.append(prediction)
        if actual_return is not None:
            self.return_history.append(actual_return)
    
    def estimate_volatility(self, returns):
        """Estimate volatility using EWMA"""
        if len(returns) < 2:
            return 0.01  # Default volatility
        
        if self.vol_ewma is None:
            self.vol_ewma = np.std(returns)
        
        # Update EWMA volatility
        recent_vol = np.std(returns[-20:]) if len(returns) >= 20 else np.std(returns)
        self.vol_ewma = self.alpha * self.vol_ewma + (1 - self.alpha) * recent_vol
        
        return self.vol_ewma
    
    def calculate_vol_penalty(self, signal, market_vol):
        """
        Calculate volatility penalty based on expected strategy volatility
        """
        if market_vol == 0:
            return 1.0
        
        # Estimate strategy volatility based on signal magnitude
        # Higher signals = higher exposure = higher volatility
        expected_strategy_vol = market_vol * abs(signal - 1.0) * 1.5
        
        # Calculate penalty if we exceed target volatility ratio
        vol_ratio = expected_strategy_vol / market_vol if market_vol > 0 else 0
        excess_vol = max(0, vol_ratio - self.target_vol_ratio)
        
        return 1.0 + excess_vol
    
    def calculate_confidence_score(self, prediction, market_vol):
        """
        Calculate confidence score based on prediction magnitude and market conditions
        """
        # Base confidence from prediction strength
        pred_strength = abs(prediction)
        base_confidence = np.tanh(pred_strength * 5)  # Scale to [0, 1]
        
        # Adjust for market volatility - be more conservative in high vol
        if market_vol > 0.02:  # High volatility regime
            vol_adjustment = 0.7
        elif market_vol > 0.01:  # Medium volatility
            vol_adjustment = 0.85
        else:  # Low volatility
            vol_adjustment = 1.0
        
        return base_confidence * vol_adjustment
    
    def process_signal(self, raw_prediction, market_returns=None):
        """
        Process raw prediction into optimized signal
        """
        # Update history
        self.update_history(raw_prediction)
        if market_returns is not None:
            self.return_history.append(market_returns)
        
        # Estimate current market volatility
        if len(self.return_history) > 0:
            market_vol = self.estimate_volatility(list(self.return_history))
        else:
            market_vol = 0.015  # Default market volatility
        
        # Calculate confidence
        confidence = self.calculate_confidence_score(raw_prediction, market_vol)
        
        # Base signal using tanh transformation
        base_signal = 1.0 + np.tanh(raw_prediction * 100) * confidence
        
        # Apply volatility penalty
        vol_penalty = self.calculate_vol_penalty(base_signal, market_vol)
        
        # Adjust signal based on penalty
        adjusted_signal = base_signal / vol_penalty
        
        # Apply regime-based adjustment
        if market_vol > 0.025:  # High volatility - reduce exposure
            adjusted_signal = 1.0 + (adjusted_signal - 1.0) * 0.6
        
        # Clip to valid range
        final_signal = np.clip(adjusted_signal, MIN_SIGNAL, MAX_SIGNAL)
        
        return final_signal


def convert_ret_to_signal_v2(ret_value, confidence=1.0, market_vol=0.015):
    """
    Enhanced signal conversion with volatility awareness
    """
    # Base signal transformation
    base_signal = 1.0 + np.tanh(ret_value * 100) * confidence
    
    # Calculate volatility penalty
    vol_penalty = 1.0
    if market_vol > 0:
        expected_exposure = abs(base_signal - 1.0)
        vol_ratio = expected_exposure * market_vol / (market_vol * 0.5)  # Baseline exposure
        excess_vol = max(0, vol_ratio - 1.2)
        vol_penalty = 1.0 + excess_vol * 0.5
    
    # Apply penalty
    adjusted_signal = base_signal / vol_penalty
    
    # Dynamic clipping based on volatility regime
    if market_vol > 0.025:  # High volatility regime
        max_signal = 1.5  # Reduce max exposure
    else:
        max_signal = MAX_SIGNAL
    
    return np.clip(adjusted_signal, MIN_SIGNAL, max_signal)


def uncertainty_aware_signal_v2(prediction, uncertainty=None, processor=None):
    """
    Enhanced uncertainty-aware signal generation
    """
    if processor is None:
        processor = AdaptiveSignalProcessor()
    
    # Process through adaptive signal processor
    signal = processor.process_signal(prediction)
    
    # Additional uncertainty adjustment if provided
    if uncertainty is not None:
        # Higher uncertainty = move signal closer to neutral (1.0)
        uncertainty_factor = np.exp(-uncertainty * 2)
        signal = 1.0 + (signal - 1.0) * uncertainty_factor
    
    return np.clip(signal, MIN_SIGNAL, MAX_SIGNAL)


# Global processor instance to maintain state across predictions
global_processor = AdaptiveSignalProcessor(lookback=180, target_vol_ratio=1.2)


def predict(test: pl.DataFrame) -> float:
    """
    Improved prediction function with enhanced signal processing
    """
    try:
        test_df = test.to_pandas()
        
        test_processed = preprocessing(test_df, 'inference')
        LGBM_R_y_pred = np.float64(LGBM_R_model.predict(test_processed)[0])
        
        # Enhanced signal processing with global processor
        signal = global_processor.process_signal(LGBM_R_y_pred)
        
        # Alternative: use the uncertainty-aware version
        # signal = uncertainty_aware_signal_v2(LGBM_R_y_pred, processor=global_processor)
        
        print(f"Raw prediction: {LGBM_R_y_pred:.6f}, Signal: {signal:.6f}")
        return np.float64(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0  # Neutral position on error


# For offline optimization (similar to the provided optimization code)
def optimize_signals_offline(predictions, train_df, lookback=180):
    """
    Offline signal optimization using the scoring metric
    """
    def objective(signals):
        solution = train_df[-lookback:].copy()
        submission = pd.DataFrame({'prediction': signals.clip(0, 2)}, index=solution.index)
        
        # Calculate strategy returns
        solution['position'] = submission['prediction']
        solution['strategy_returns'] = \
            solution['risk_free_rate'] * (1 - solution['position']) + \
            solution['forward_returns'] * solution['position']
        
        # Calculate Sharpe with penalties
        strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
        strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
        strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
        strategy_std = solution['strategy_returns'].std()
        
        if strategy_std == 0:
            return 1e6
        
        sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(252)
        
        # Volatility penalty
        strategy_volatility = strategy_std * np.sqrt(252)
        market_std = solution['forward_returns'].std()
        market_volatility = market_std * np.sqrt(252)
        
        excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
        vol_penalty = 1 + excess_vol
        
        adjusted_sharpe = sharpe / vol_penalty
        
        return -adjusted_sharpe  # Minimize negative Sharpe
    
    # Initialize with processed predictions
    x0 = np.array([convert_ret_to_signal_v2(p) for p in predictions])
    
    # Optimize
    result = minimize(
        objective, 
        x0, 
        method='Powell',
        bounds=Bounds(lb=MIN_SIGNAL, ub=MAX_SIGNAL),
        options={'maxiter': 1000, 'ftol': 1e-8}
    )
    
    return result.x


# Inference server setup
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/hull-tactical-market-prediction/',)
    )

Raw prediction: -0.005964, Signal: 0.986458
Raw prediction: -0.005427, Signal: 0.988584
Raw prediction: 0.005419, Signal: 1.011384
Raw prediction: 0.007653, Signal: 1.020943
Raw prediction: -0.002896, Signal: 0.996532
Raw prediction: 0.002457, Signal: 1.002514
Raw prediction: 0.002311, Signal: 1.002231
Raw prediction: 0.002891, Signal: 1.003456
Raw prediction: 0.008308, Signal: 1.024030
Raw prediction: 0.000099, Signal: 1.000004
